In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
from keras import Sequential, Input, layers
from keras.applications.efficientnet_v2 import EfficientNetV2B0

In [3]:
def load_train_df():
    df = pd.read_csv("../../datasets/food-101-100-images/meta/train.txt", header=None)
    return df

In [4]:
def load_test_df():
    df = pd.read_csv("../../datasets/food-101-100-images/meta/test.txt", header=None)
    return df

In [5]:
load_train_df()

,0
0,apple_pie/1005649
1,apple_pie/1014775
2,apple_pie/1026328
3,apple_pie/1028787
4,apple_pie/1043283
...,...
7586,waffles/1343456
7587,waffles/1351305
7588,waffles/1353542
7589,waffles/1354919


In [6]:
load_test_df()

,0
0,apple_pie/1011328
1,apple_pie/101251
2,apple_pie/1034399
3,apple_pie/103801
4,apple_pie/1038694
...,...
2504,waffles/1311578
2505,waffles/1313096
2506,waffles/1336006
2507,waffles/1343475


# Parse image

In [28]:
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

def parse_image(df):
    images = []
    labels = []
    for path in df[0]:
        class_name = Path(path).parent.name
        img_path = f"../../datasets/food-101/images/{path}.jpg"
        img = Image.open(img_path).convert('RGB').resize((224,224))
        images.append(np.array(img, dtype=np.float32))
        labels.append(class_name)

    le = LabelEncoder()
    y = le.fit_transform(labels)

    X = np.stack(images)
    return X, y, le

In [29]:
train_X, train_y, le = parse_image(load_train_df())

In [9]:
train_X.shape

(7591, 224, 224, 3)

In [10]:
def efficient_net_model():
    model = EfficientNetV2B0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    model.trainable = False

    return model

### Load single image from datasets/predict_images and run model.predict

In [14]:
def initialise_model():
    model = Sequential()

    model.add(Input(shape=(224,224,3)))
    model.add(efficient_net_model())

    model.add(layers.Flatten())

    model.add(layers.Dense(100, activation='relu'))
    model.add(layers.Dense(200, activation='relu'))

    model.add(layers.Dense(101, activation='softmax'))

    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    return model

In [15]:
model = initialise_model()
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetv2-b0 (Functional)  │ (None, 7, 7, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 62720)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 100)            │     6,272,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 200)            │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 101)            │        20,301 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,231,913 (46.66 MB)

 Trainable params: 6,312,601 (24.08 MB)

 Non-trainable params: 5,919,312 (22.58 MB)

In [16]:
model.fit(train_X, train_y, epochs=10)

238/238 ━━━━━━━━━━━━━━━━━━━━ 39s 150ms/step - accuracy: 0.0075 - loss: 4.7445


In [34]:
# 1. Select a sample image path from datasets/predict_images
img_path = "../../datasets/predict_images/apple_pie/apple_pie_2.jpg"

img = Image.open(img_path).convert('RGB').resize((224,224))
img_array = np.array(img, dtype=np.float32)

# 3. Add batch dimension -> shape: (1, height, width, 3)
img_batch = np.expand_dims(img_array, axis=0)
predictions = model.predict(img_batch)

predicted_class_int = np.argmax(predictions[0])

   # Get the probability/confidence for that class (e.g. 0.85 or 85%)
confidence = predictions[0][predicted_class_int]

   # Convert integer back to class name (e.g. 'baklava')
predicted_class_name = le.inverse_transform([predicted_class_int])[0]

print(f"Integer label: {predicted_class_int}")
print(f"Class name: {predicted_class_name} ({confidence * 100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Integer label: 44
Class name: fried_rice (99.99%)
